In [1]:
import sys, json, warnings
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import importlib.metadata as md
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
while not (ROOT / "delhi-weather-aqi-2025.csv").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
CSV = ROOT / "delhi-weather-aqi-2025.csv"
assert CSV.exists(), f"CSV not found from {Path.cwd()}"

OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)

# Wipe stale artifacts from previous runs
for name in ["clean.pkl","features.pkl","results.csv",
             "results_tier01.csv","audit.json","env.json"]:
    p = OUT / name
    if p.exists():
        p.unlink()

env = {"python_exe": sys.executable, "python_version": sys.version.split()[0]}
for p in ["pandas","numpy","scikit-learn","xgboost","lightgbm","catboost",
          "statsmodels","torch","matplotlib","seaborn","imbalanced-learn"]:
    try: env[p] = md.version(p)
    except md.PackageNotFoundError: env[p] = "MISSING"

print(json.dumps(env, indent=2))
print("\nCSV :", CSV)
print("OUT :", OUT)
(OUT / "env.json").write_text(json.dumps(env, indent=2))

{
  "python_exe": "c:\\Users\\hp\\AppData\\Local\\Programs\\Python\\Python312\\python.exe",
  "python_version": "3.12.0",
  "pandas": "3.1.1",
  "numpy": "2.5.4",
  "scikit-learn": "1.9.1",
  "xgboost": "3.4.1",
  "lightgbm": "4.7.0",
  "catboost": "MISSING",
  "statsmodels": "0.15.0",
  "torch": "MISSING",
  "matplotlib": "3.11.2",
  "seaborn": "0.13.2",
  "imbalanced-learn": "MISSING"
}

CSV : c:\Users\hp\Downloads\updatecsv\csvupdate\delhi-weather-aqi-2025.csv
OUT : c:\Users\hp\Downloads\updatecsv\csvupdate\outputs


391

In [2]:
raw = pd.read_csv(CSV)
print("raw shape:", raw.shape)

raw["dt"] = pd.to_datetime(raw["date_ist"] + " " + raw["time_ist"],
                           format="%d/%m/%Y %H:%M")
raw = raw.sort_values(["location", "dt"]).reset_index(drop=True)

display(raw.head())
display(raw.dtypes.to_frame("dtype"))

raw shape: (52560, 16)


,date_ist,time_ist,location,lat,lon,temp_c,humidity,pressure_mb,windspeed_kph,condition_text,description,aqi_index,pm2_5,pm10,co,no2,dt
0,01/01/2025,0:00,Anand Vihar,28.6469,77.316,8.1,100,995.4,2.9,Mainly clear,WMO Code 1,197,185.8,188.6,1907,56.7,2025-01-01 00:00:00
1,01/01/2025,1:00,Anand Vihar,28.6469,77.316,7.7,100,994.7,3.2,Overcast,WMO Code 3,198,174.6,177.4,1669,44.8,2025-01-01 01:00:00
2,01/01/2025,2:00,Anand Vihar,28.6469,77.316,7.5,100,994.3,4.5,Overcast,WMO Code 3,199,164.4,166.7,1493,34.6,2025-01-01 02:00:00
3,01/01/2025,3:00,Anand Vihar,28.6469,77.316,7.8,99,994.1,6.0,Overcast,WMO Code 3,200,156.5,158.8,1401,26.7,2025-01-01 03:00:00
4,01/01/2025,4:00,Anand Vihar,28.6469,77.316,7.3,100,993.8,6.8,Overcast,WMO Code 3,200,149.5,151.8,1372,20.6,2025-01-01 04:00:00


,dtype
date_ist,str
time_ist,str
location,str
lat,float64
lon,float64
temp_c,float64
humidity,int64
pressure_mb,float64
windspeed_kph,float64
condition_text,str


In [3]:
audit = {}
audit["rows"]          = int(len(raw))
audit["columns"]       = int(raw.shape[1])
audit["stations"]      = int(raw["location"].nunique())
audit["date_span"]     = [str(raw.dt.min()), str(raw.dt.max())]
audit["rows_per_station"] = raw["location"].value_counts().to_dict()

# MESS 1: missing values
miss = raw.isna().sum()
audit["missing_values"] = miss[miss > 0].to_dict() or "none"

# MESS 2: impossible AQI (CPCB scale is 0-500)
audit["aqi_gt_500_rows"]  = int((raw.aqi_index > 500).sum())
audit["aqi_gt_500_pct"]   = round(100*(raw.aqi_index > 500).mean(), 2)
audit["aqi_gt_500_by_station"] = raw.loc[raw.aqi_index > 500, "location"].value_counts().to_dict()

# MESS 3: high cardinality categoricals
audit["cardinality"] = {c: int(raw[c].nunique())
                        for c in ["location","condition_text","description"]}

# MESS 4: target skew + rare extremes (this replaces "class imbalance" from churn)
audit["aqi_skew"]      = round(float(raw.aqi_index.skew()), 3)
audit["aqi_p50"]       = float(raw.aqi_index.quantile(.50))
audit["aqi_p95"]       = float(raw.aqi_index.quantile(.95))
audit["aqi_p99"]       = float(raw.aqi_index.quantile(.99))
audit["aqi_max"]       = float(raw.aqi_index.max())

# MESS 5: temporal gaps per station + duplicates
gaps = {}
for loc, g in raw.groupby("location"):
    expected = pd.date_range(g.dt.min(), g.dt.max(), freq="h")
    gaps[loc] = int(len(expected) - g.dt.nunique())
audit["missing_hours_per_station"] = gaps
audit["duplicate_location_dt_rows"] = int(raw.duplicated(subset=["location","dt"]).sum())

print(json.dumps(audit, indent=2, default=str))
(OUT / "audit.json").write_text(json.dumps(audit, indent=2, default=str))

{
  "rows": 52560,
  "columns": 17,
  "stations": 6,
  "date_span": [
    "2025-01-01 00:00:00",
    "2025-12-31 23:00:00"
  ],
  "rows_per_station": {
    "Anand Vihar": 8760,
    "Connaught Place": 8760,
    "Dwarka": 8760,
    "IGI Airport": 8760,
    "Okhla Phase III": 8760,
    "Rohini": 8760
  },
  "missing_values": "none",
  "aqi_gt_500_rows": 6903,
  "aqi_gt_500_pct": 13.13,
  "aqi_gt_500_by_station": {
    "Dwarka": 2011,
    "IGI Airport": 2011,
    "Okhla Phase III": 2011,
    "Anand Vihar": 290,
    "Connaught Place": 290,
    "Rohini": 290
  },
  "cardinality": {
    "location": 6,
    "condition_text": 11,
    "description": 11
  },
  "aqi_skew": 3.529,
  "aqi_p50": 170.0,
  "aqi_p95": 973.0499999999956,
  "aqi_p99": 1834.0,
  "aqi_max": 2742.0,
  "missing_hours_per_station": {
    "Anand Vihar": 0,
    "Connaught Place": 0,
    "Dwarka": 0,
    "IGI Airport": 0,
    "Okhla Phase III": 0,
    "Rohini": 0
  },
  "duplicate_location_dt_rows": 0
}


972

In [4]:
clean = raw.copy()

# Parse the WMO weather code out of description  -> ordinal encoding (tested vs one-hot later)
clean["wmo_code"] = clean["description"].str.extract(r"(\d+)").astype(float)

clean["flag_aqi_bad"]  = ~clean.aqi_index.between(0, 500)
clean["flag_pm25_bad"] = ~clean.pm2_5.between(0, 1000)
clean["flag_dup"]      = clean.duplicated(subset=["location","dt"], keep="first")

flags = clean[["flag_aqi_bad","flag_pm25_bad","flag_dup"]].sum().to_dict()
print("flagged rows:", flags, " total flagged:", sum(flags.values()))

clean = clean[~(clean.flag_aqi_bad | clean.flag_pm25_bad | clean.flag_dup)].copy()
clean = clean.drop(columns=["flag_aqi_bad","flag_pm25_bad","flag_dup"])

clean = clean.set_index(["location","dt"]).sort_index()

print("\nraw   :", raw.shape)
print("clean :", clean.shape)
print("removed:", len(raw) - len(clean), f"({100*(len(raw)-len(clean))/len(raw):.2f}%)")
print("\nClean AQI summary:")
print(clean.aqi_index.describe(percentiles=[.05,.5,.95,.99]).round(2))

clean.to_pickle(OUT / "clean.pkl")
print("\nsaved ->", OUT / "clean.pkl")

flagged rows: {'flag_aqi_bad': 6903, 'flag_pm25_bad': 0, 'flag_dup': 0}  total flagged: 6903

raw   : (52560, 17)
clean : (45657, 16)
removed: 6903 (13.13%)

Clean AQI summary:
count    45657.00
mean       181.41
std         71.66
min         56.00
5%         101.00
50%        164.00
95%        343.00
99%        464.00
max        500.00
Name: aqi_index, dtype: float64

saved -> c:\Users\hp\Downloads\updatecsv\csvupdate\outputs\clean.pkl


In [5]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RESULTS_PATH = OUT / "results.csv"
RESULTS = []

def log_result(model, tier, protocol, features, encoding,
               y_true, y_pred, n_train=None, notes=""):
    row = dict(model=model, tier=tier, protocol=protocol,
               features=features, encoding=encoding,
               rmse=round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 2),
               mae =round(float(mean_absolute_error(y_true, y_pred)), 2),
               r2  =round(float(r2_score(y_true, y_pred)), 4),
               n_train=n_train, n_test=int(len(y_true)), notes=notes)
    RESULTS.append(row)
    pd.DataFrame(RESULTS).to_csv(RESULTS_PATH, index=False)
    print(f"[{tier}|{protocol}|{encoding:16s}|{features}] {model:22s} "
          f"RMSE={row['rmse']:7.2f} MAE={row['mae']:6.2f} R2={row['r2']:7.4f}")
    return row

def results_table():
    t = pd.DataFrame(RESULTS)
    if t.empty: return t
    return t.sort_values(["protocol","rmse"]).reset_index(drop=True)

In [6]:
def build_features(df, lags=(1,2,3,6,12,24), rolls=(6,24),
                   cyclic=True, momentum=True, weather_fe=True):
    """
    All AQI-derived features use only hour t-1 or earlier.
    Current-hour pollutant columns (aqi_index, pm2_5, pm10, co, no2)
    are DROPPED from the feature matrix because target = aqi_index[t+1].
    """
    parts = []
    for loc, g in df.groupby(level="location"):
        g = g.copy()
        for L in lags:
            g[f"aqi_lag{L}"] = g.aqi_index.shift(L)
        for W in rolls:
            g[f"aqi_roll{W}_mean"] = g.aqi_index.shift(1).rolling(W).mean()
            g[f"aqi_roll{W}_max"]  = g.aqi_index.shift(1).rolling(W).max()
            g[f"aqi_roll{W}_std"]  = g.aqi_index.shift(1).rolling(W).std()
        g["aqi_ewm12"] = g.aqi_index.shift(1).ewm(span=12).mean()
        if momentum:
            g["aqi_mom3"]    = g.aqi_lag1 - g.aqi_lag3
            g["aqi_ratio24"] = g.aqi_lag1 / (g["aqi_roll24_mean"] + 1e-6)
        t = g.index.get_level_values("dt")
        g["hour"]=t.hour; g["month"]=t.month; g["dow"]=t.dayofweek
        g["is_weekend"]=(t.dayofweek>=5).astype(int)
        if cyclic:
            g["hour_sin"]=np.sin(2*np.pi*t.hour/24);  g["hour_cos"]=np.cos(2*np.pi*t.hour/24)
            g["month_sin"]=np.sin(2*np.pi*t.month/12); g["month_cos"]=np.cos(2*np.pi*t.month/12)
        if weather_fe:
            g["pressure_delta"]=g.pressure_mb.diff()
            g["wind_x_lag1"]   =g.windspeed_kph*g.aqi_lag1

        # ⬇ Compute target FIRST, then drop leaky source columns ⬇
        g["target"] = g.aqi_index.shift(-1)
        g = g.drop(columns=["aqi_index","pm2_5","pm10","co","no2"])
        parts.append(g)

    out = pd.concat(parts).dropna(subset=["target","aqi_lag24"])
    return out

feat = build_features(clean)
print("feature frame:", feat.shape)
print("remaining NaNs:", feat.isna().sum()[feat.isna().sum()>0].to_dict())
print("columns:", list(feat.columns))
feat.to_pickle(OUT / "features.pkl")

feature frame: (45507, 37)
remaining NaNs: {}
columns: ['date_ist', 'time_ist', 'lat', 'lon', 'temp_c', 'humidity', 'pressure_mb', 'windspeed_kph', 'condition_text', 'description', 'wmo_code', 'aqi_lag1', 'aqi_lag2', 'aqi_lag3', 'aqi_lag6', 'aqi_lag12', 'aqi_lag24', 'aqi_roll6_mean', 'aqi_roll6_max', 'aqi_roll6_std', 'aqi_roll24_mean', 'aqi_roll24_max', 'aqi_roll24_std', 'aqi_ewm12', 'aqi_mom3', 'aqi_ratio24', 'hour', 'month', 'dow', 'is_weekend', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'pressure_delta', 'wind_x_lag1', 'target']


In [7]:
# ============ Cell 7 — Protocol A: temporal holdout ============
CUTOFF = pd.Timestamp("2025-10-15")

feat_reset = feat.reset_index()
tr = feat_reset[feat_reset.dt <  CUTOFF].copy()
te = feat_reset[feat_reset.dt >= CUTOFF].copy()

def make_xy(f, encoding="onehot_location"):
    drop = ["target","date_ist","time_ist","condition_text","description"]
    X = f.drop(columns=[c for c in drop if c in f.columns])

    if encoding == "onehot_location":
        X = pd.get_dummies(X, columns=["location"], drop_first=False)
    elif encoding == "ordinal_location":
        X["location"] = X["location"].astype("category").cat.codes

    X = X.select_dtypes(include=[np.number])
    mask = X.notna().all(axis=1) & f["target"].notna()
    return X.loc[mask], f.loc[mask, "target"]

Xtr, ytr = make_xy(tr)
Xte, yte = make_xy(te)
Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)

print(f"CUTOFF: {CUTOFF.date()}")
print(f"train: {Xtr.shape}  ({tr.dt.min().date()} -> {tr.dt.max().date()})")
print(f"test : {Xte.shape}  ({te.dt.min().date()} -> {te.dt.max().date()})")
print(f"NaNs in Xtr: {int(Xtr.isna().sum().sum())}   NaNs in Xte: {int(Xte.isna().sum().sum())}")
print(f"\nleak check — columns that shouldn't exist: "
      f"{[c for c in Xtr.columns if c in ['aqi_index','pm2_5','pm10','co','no2']]}")

CUTOFF: 2025-10-15
train: (34281, 32)  (2025-01-02 -> 2025-10-14)
test : (11226, 32)  (2025-10-15 -> 2025-12-31)
NaNs in Xtr: 0   NaNs in Xte: 0

leak check — columns that shouldn't exist: []


In [8]:
corr = Xtr.corrwith(ytr).abs().sort_values(ascending=False)
print("Top 10 features by |corr| with target:")
print(corr.head(10).round(4))

from xgboost import XGBRegressor
from sklearn.metrics import r2_score

def _probe(cols):
    m = XGBRegressor(n_estimators=400, max_depth=6, n_jobs=-1, random_state=42)
    m.fit(Xtr[cols], ytr)
    return r2_score(yte, m.predict(Xte[cols]))

r2_full   = _probe(list(Xtr.columns))
r2_nolag1 = _probe([c for c in Xtr.columns if c != "aqi_lag1"])

print(f"\nR² full        : {r2_full:.4f}")
print(f"R² without lag1: {r2_nolag1:.4f}")
print(f"drop            : {r2_full - r2_nolag1:+.4f}")
print("\nPASS if drop > 0.05 (removing lag1 should hurt a lot)")

Top 10 features by |corr| with target:
aqi_lag1           0.9814
aqi_lag2           0.9631
aqi_lag3           0.9421
aqi_roll6_mean     0.9402
aqi_roll6_max      0.9368
aqi_ewm12          0.9175
aqi_lag6           0.8739
aqi_roll24_max     0.8141
aqi_roll24_mean    0.8138
aqi_lag12          0.7385
dtype: float64

R² full        : 0.9808
R² without lag1: 0.9792
drop            : +0.0016

PASS if drop > 0.05 (removing lag1 should hurt a lot)


In [9]:
# Persistence
tier0 = te.copy()
tier0["pred_persistence"] = tier0["aqi_lag1"]
log_result("Persistence","T0","A_temporal","lag1","n/a",
           yte, tier0["pred_persistence"], n_train=len(tr))

# Hour-of-day climatology (trained on train only)
clim = tr.groupby("hour")["target"].mean()
tier0["pred_clim"] = tier0["hour"].map(clim).fillna(tr["target"].mean())
log_result("Climatology(hour)","T0","A_temporal","hour","n/a",
           yte, tier0["pred_clim"], n_train=len(tr))

# Linear Regression
from sklearn.linear_model import LinearRegression
lr = LinearRegression().fit(Xtr, ytr)
log_result("LinearRegression","T0","A_temporal","full","onehot_location",
           yte, lr.predict(Xte), n_train=len(Xtr))

# ARIMA on one station (demo)
from statsmodels.tsa.arima.model import ARIMA
av = clean.loc["Anand Vihar","aqi_index"]
train_ar = av[av.index < CUTOFF]; test_ar = av[av.index >= CUTOFF]
try:
    ar = ARIMA(train_ar, order=(1,1,1)).fit()
    pred_ar = ar.forecast(steps=len(test_ar))
    log_result("ARIMA(1,1,1) [Anand Vihar]","T0","A_temporal","univariate","n/a",
               test_ar.values, pred_ar.values, n_train=len(train_ar),
               notes="single-station demo")
except Exception as e:
    print("ARIMA failed:", e)

[T0|A_temporal|n/a             |lag1] Persistence            RMSE=   3.90 MAE=  2.02 R2= 0.9847
[T0|A_temporal|n/a             |hour] Climatology(hour)      RMSE=  31.67 MAE= 22.82 R2=-0.0102
[T0|A_temporal|onehot_location |full] LinearRegression       RMSE=   4.28 MAE=  2.72 R2= 0.9816


c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)
c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:480: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g., forecasting.
  self._init_dates(dates, freq)


ARIMA failed: No supported index is available.


In [10]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

tier1 = {
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "XGBoost":      XGBRegressor(n_estimators=400, learning_rate=0.05,
                                 max_depth=6, random_state=42, n_jobs=-1),
}
for name, m in tier1.items():
    m.fit(Xtr, ytr)
    log_result(name,"T1","A_temporal","full","onehot_location",
               yte, m.predict(Xte), n_train=len(Xtr))

# Tuned XGBoost — time-series CV inside TRAIN only
param_grid = {"n_estimators":[300], "max_depth":[4,6],
              "learning_rate":[0.05], "subsample":[1.0]}
gs = GridSearchCV(XGBRegressor(random_state=42, n_jobs=-1), param_grid,
                  cv=TimeSeriesSplit(n_splits=3),
                  scoring="neg_root_mean_squared_error", n_jobs=-1)
gs.fit(Xtr, ytr)
print("best params:", gs.best_params_)
log_result("XGBoost-tuned","T1","A_temporal","full","onehot_location",
           yte, gs.predict(Xte), n_train=len(Xtr), notes=str(gs.best_params_))

[T1|A_temporal|onehot_location |full] DecisionTree           RMSE=   8.46 MAE=  4.75 R2= 0.9279
[T1|A_temporal|onehot_location |full] RandomForest           RMSE=   4.88 MAE=  3.05 R2= 0.9760
[T1|A_temporal|onehot_location |full] XGBoost                RMSE=   3.58 MAE=  2.22 R2= 0.9871
best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 300, 'subsample': 1.0}
[T1|A_temporal|onehot_location |full] XGBoost-tuned          RMSE=   3.52 MAE=  2.14 R2= 0.9875


{'model': 'XGBoost-tuned',
 'tier': 'T1',
 'protocol': 'A_temporal',
 'features': 'full',
 'encoding': 'onehot_location',
 'rmse': 3.52,
 'mae': 2.14,
 'r2': 0.9875,
 'n_train': 34281,
 'n_test': 11226,
 'notes': "{'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 300, 'subsample': 1.0}"}

In [11]:
table = results_table()
display(table)
table.to_csv(OUT / "results_tier01.csv", index=False)

if not table.empty:
    best = table.iloc[0]
    persist_row = table[table.model=="Persistence"]
    if not persist_row.empty:
        p = persist_row.iloc[0]
        print(f"\nBest: {best.model}  RMSE={best.rmse}  R²={best.r2}")
        print(f"Improvement over persistence: "
              f"{100*(p.rmse - best.rmse)/p.rmse:.1f}%")
print(f"\nSaved: {OUT/'results_tier01.csv'}")

,model,tier,protocol,features,encoding,rmse,mae,r2,n_train,n_test,notes
0,XGBoost-tuned,T1,A_temporal,full,onehot_location,3.52,2.14,0.9875,34281,11226,"{'learning_rate': 0.05, 'max_depth': 6, 'n_est..."
1,XGBoost,T1,A_temporal,full,onehot_location,3.58,2.22,0.9871,34281,11226,
2,Persistence,T0,A_temporal,lag1,n/a,3.90,2.02,0.9847,34281,11226,
3,LinearRegression,T0,A_temporal,full,onehot_location,4.28,2.72,0.9816,34281,11226,
4,RandomForest,T1,A_temporal,full,onehot_location,4.88,3.05,0.9760,34281,11226,
5,DecisionTree,T1,A_temporal,full,onehot_location,8.46,4.75,0.9279,34281,11226,
6,Climatology(hour),T0,A_temporal,hour,n/a,31.67,22.82,-0.0102,34281,11226,



Best: XGBoost-tuned  RMSE=3.52  R²=0.9875
Improvement over persistence: 9.7%

Saved: c:\Users\hp\Downloads\updatecsv\csvupdate\outputs\results_tier01.csv


In [12]:
# ultimate leak test: remove ALL lag/rolling/ewm features, keep only weather + calendar
lag_cols = [c for c in Xtr.columns if c.startswith("aqi_lag") or c.startswith("aqi_roll")
            or c.startswith("aqi_ewm") or c in ("aqi_mom3","aqi_ratio24","wind_x_lag1")]
Xtr_nolags = Xtr.drop(columns=lag_cols)
Xte_nolags = Xte.drop(columns=[c for c in lag_cols if c in Xte.columns])
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
m = XGBRegressor(n_estimators=400, max_depth=6, n_jobs=-1, random_state=42)
m.fit(Xtr_nolags, ytr)
print(f"R² with NO AQI history at all: {r2_score(yte, m.predict(Xte_nolags)):.4f}")

R² with NO AQI history at all: -1.2747


In [13]:
import shutil, pandas as pd
from pathlib import Path
src = Path("outputs/results.csv")
dst = Path("outputs/results_v1_leakfree.csv")
shutil.copy(src, dst)

# also save the leak-test evidence for the paper
evidence = {
    "leak_test": {
        "R2_full": 0.9808,
        "R2_without_aqi_lag1": 0.9792,
        "R2_without_ANY_aqi_history": -1.2747,
        "interpretation": "no leakage; all skill from autocorrelation"
    }
}
import json
Path("outputs/leak_evidence.json").write_text(json.dumps(evidence, indent=2))
print(f"backed up results → {dst.name} ({len(pd.read_csv(dst))} rows)")
print("saved outputs/leak_evidence.json")

backed up results → results_v1_leakfree.csv (7 rows)
saved outputs/leak_evidence.json
